## Step 1 — Inspect the Gold and Predictions tables

In [0]:
PREDICTION_TABLE = (
    "predictivemaintenancecatalog.gold_data.test_failure_predictions"
)

GOLD_TABLE = (
    "predictivemaintenancecatalog.gold_data.streaming"
)

print("PREDICTIONS SCHEMA")
spark.table(PREDICTION_TABLE).printSchema()

print("\nGOLD SCHEMA")
spark.table(GOLD_TABLE).printSchema()

## Step 2 — Combine Gold and Predictions

### Check for duplicate machine IDs

In [0]:
predictions_df = spark.table(PREDICTION_TABLE)
gold_df = spark.table(GOLD_TABLE)

for name, df in [
    ("Predictions", predictions_df),
    ("Gold", gold_df)
]:
    print(f"{name}:")
    print("Total rows:", df.count())
    print("Unique UDI:", df.select("UDI").distinct().count())
    print()

### Join the tables

In [0]:
from pyspark.sql import functions as F

monitoring_df = (
    predictions_df.alias("p")
    .join(
        gold_df.alias("g"),
        on="UDI",
        how="left"
    )
)

display(monitoring_df.limit(10))

### Verify the join

In [0]:
print("Predictions:", predictions_df.count())
print("Monitoring rows:", monitoring_df.count())

missing_gold = monitoring_df.filter(
    F.col("Air_temperature_K").isNull()
).count()

print("Records missing Gold features:", missing_gold)

## Identify CSV batches automatically

In [0]:
from pyspark.sql import functions as F

csv_inbox = (
    "/Volumes/predictivemaintenancecatalog/raw_data/"
    "landing/csv_inbox"
)

landing_schema = spark.table(
    "predictivemaintenancecatalog.raw_data.landing_testdata"
).schema

# Read CSV filenames and corresponding machine IDs
csv_batches = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("pathGlobFilter", "*.csv")
    .schema(landing_schema)
    .load(csv_inbox)
    .select(
        "UDI",
        F.col("_metadata.file_name").alias("batch")
    )
)

# Prevent duplicate UDIs from multiplying monitoring rows
duplicates = (
    csv_batches.groupBy("UDI")
    .count()
    .filter(F.col("count") > 1)
)

assert duplicates.limit(1).count() == 0, \
    "Duplicate UDI values found across CSV batches"

# Automatically attach source CSV filename
monitoring_df = (
    monitoring_df
    .join(csv_batches, on="UDI", how="left")
    .withColumn(
        "batch",
        F.coalesce(
            F.col("batch"),
            F.lit("Other / Pre-CSV Data")
        )
    )
)

display(
    monitoring_df.groupBy("batch")
    .count()
    .orderBy("batch")
)

## Step 3 Create the Delta Monitoring Table

### Final data integrity checks

In [0]:
from pyspark.sql import functions as F

pred_duplicates = (
    predictions_df.groupBy("UDI")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

gold_duplicates = (
    gold_df.groupBy("UDI")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

missing_labels = monitoring_df.filter(
    F.col("Machine_failure").isNull()
).count()

print("Duplicate prediction UDIs:", pred_duplicates)
print("Duplicate Gold UDIs:", gold_duplicates)
print("Missing actual labels:", missing_labels)

### Create the monitoring Delta table

In [0]:
assert pred_duplicates == 0
assert gold_duplicates == 0
assert missing_labels == 0

from delta.tables import DeltaTable
from pyspark.sql import functions as F

MONITOR_TABLE = (
    "predictivemaintenancecatalog.gold_data.prediction_monitoring"
)

# Add the batch column to the existing table if missing
if "batch" not in spark.table(MONITOR_TABLE).columns:
    spark.sql(f"""
        ALTER TABLE {MONITOR_TABLE}
        ADD COLUMNS (batch STRING)
    """)

# Ensure each UDI is unique before merging
duplicate_ids = (
    monitoring_df.groupBy("UDI")
    .count()
    .filter(F.col("count") > 1)
)

assert duplicate_ids.limit(1).count() == 0, \
    "Duplicate UDI values detected"

# Update existing records and insert new ones
delta_table = DeltaTable.forName(spark, MONITOR_TABLE)

(
    delta_table.alias("target")
    .merge(
        monitoring_df.alias("source"),
        "target.UDI = source.UDI"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Monitoring table updated successfully.")

### Verify the new table

In [0]:
display(
    spark.table(MONITOR_TABLE)
    .groupBy("batch")
    .count()
    .orderBy("batch")
)

In [0]:
spark.table(MONITOR_TABLE).display()

## Step 4: Model Performance Monitoring

In [0]:
from pyspark.sql import functions as F

MONITOR_TABLE = (
    "predictivemaintenancecatalog.gold_data.prediction_monitoring"
)

df = spark.table(MONITOR_TABLE)

actual = F.col("Machine_failure")
predicted = F.col("predicted_failure")

metrics = df.agg(
    F.count("*").alias("Total"),

    F.sum(((actual == 1) & (predicted == 1)).cast("int")).alias("TP"),
    F.sum(((actual == 1) & (predicted == 0)).cast("int")).alias("FN"),
    F.sum(((actual == 0) & (predicted == 1)).cast("int")).alias("FP"),
    F.sum(((actual == 0) & (predicted == 0)).cast("int")).alias("TN")
).first()

tp = metrics["TP"]
fn = metrics["FN"]
fp = metrics["FP"]
tn = metrics["TN"]
total = metrics["Total"]

precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0
f2 = 5 * tp / (5 * tp + 4 * fn + fp) if (5 * tp + 4 * fn + fp) else 0
alert_rate = (tp + fp) / total if total else 0

print(f"Total Records: {total}")
print(f"True Positives: {tp}")
print(f"False Negatives: {fn}")
print(f"False Positives: {fp}")
print(f"True Negatives: {tn}")

print(f"\nPrecision: {precision:.2%}")
print(f"Recall: {recall:.2%}")
print(f"F2 Score: {f2:.4f}")
print(f"Alert Rate: {alert_rate:.2%}")

## Save the metrics to a new table 

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

METRICS_TABLE = (
    "predictivemaintenancecatalog.gold_data."
    "prediction_monitoring_metrics"
)

df = spark.table(MONITOR_TABLE)

actual = F.col("Machine_failure")
pred = F.col("predicted_failure")

# Calculate metrics for each batch and model
metrics_df = (
    df.groupBy("batch", "model_uri", "threshold")
    .agg(
        F.count("*").alias("Total"),
        F.sum(((actual == 1) & (pred == 1)).cast("int")).alias("TP"),
        F.sum(((actual == 1) & (pred == 0)).cast("int")).alias("FN"),
        F.sum(((actual == 0) & (pred == 1)).cast("int")).alias("FP"),
        F.sum(((actual == 0) & (pred == 0)).cast("int")).alias("TN")
    )
    .withColumn(
        "Precision",
        F.when(
            (F.col("TP") + F.col("FP")) > 0,
            F.col("TP") / (F.col("TP") + F.col("FP"))
        )
    )
    .withColumn(
        "Recall",
        F.when(
            (F.col("TP") + F.col("FN")) > 0,
            F.col("TP") / (F.col("TP") + F.col("FN"))
        )
    )
    .withColumn(
        "F2",
        F.when(
            (5*F.col("TP") + 4*F.col("FN") + F.col("FP")) > 0,
            5*F.col("TP") /
            (5*F.col("TP") + 4*F.col("FN") + F.col("FP"))
        )
    )
    .withColumn(
        "Alert_Rate",
        (F.col("TP") + F.col("FP")) / F.col("Total")
    )
    .withColumn("updated_at", F.current_timestamp())
)

# Create the metrics table if it doesn't exist
if not spark.catalog.tableExists(METRICS_TABLE):
    (
        metrics_df.write
        .format("delta")
        .saveAsTable(METRICS_TABLE)
    )
else:
    # Update existing batch metrics and insert new batches
    (
        DeltaTable.forName(spark, METRICS_TABLE)
        .alias("target")
        .merge(
            metrics_df.alias("source"),
            """
            target.batch = source.batch
            AND target.model_uri = source.model_uri
            AND target.threshold = source.threshold
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

display(
    spark.table(METRICS_TABLE)
    .orderBy("batch")
)

## Step 5 — Rolling model performance metrics (latest 1,000 labeled predictions)

In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

MONITOR_TABLE = "predictivemaintenancecatalog.gold_data.prediction_monitoring"
ROLLING_TABLE = "predictivemaintenancecatalog.gold_data.prediction_monitoring_metrics_rolling"
WINDOW_SIZE = 1000
MIN_FAILURES = 20  # Below this, flag recall as low-support

# Only records with known actual outcomes contribute to performance metrics.
observations = (
    spark.table(MONITOR_TABLE)
    .filter(
        F.col("Machine_failure").isin(0, 1)
        & F.col("predicted_failure").isin(0, 1)
        & F.col("UDI").isNotNull()
        & F.col("prediction_timestamp").isNotNull()
        & F.col("model_uri").isNotNull()
        & F.col("threshold").isNotNull()
    )
)

# Latest 1,000 *labeled observations* for each model / threshold.
# UDI breaks ties when a batch has the same prediction timestamp.
order = Window.partitionBy("model_uri", "threshold").orderBy(
    F.col("prediction_timestamp").desc(), F.col("UDI").desc()
)
recent = (
    observations.withColumn("_rn", F.row_number().over(order))
    .filter(F.col("_rn") <= WINDOW_SIZE)
)

actual = F.col("Machine_failure")
predicted = F.col("predicted_failure")
rolling_df = (
    recent.groupBy("model_uri", "threshold")
    .agg(
        F.count("*").alias("window_records"),
        F.min("prediction_timestamp").alias("window_start"),
        F.max("prediction_timestamp").alias("window_end"),
        F.max(F.when(F.col("_rn") == 1, F.col("UDI"))).alias("latest_udi"),
        F.sum(((actual == 1) & (predicted == 1)).cast("long")).alias("TP"),
        F.sum(((actual == 1) & (predicted == 0)).cast("long")).alias("FN"),
        F.sum(((actual == 0) & (predicted == 1)).cast("long")).alias("FP"),
        F.sum(((actual == 0) & (predicted == 0)).cast("long")).alias("TN"),
    )
    .withColumn("actual_failures", F.col("TP") + F.col("FN"))
    .withColumn(
        "precision",
        F.when(F.col("TP") + F.col("FP") > 0,
               F.col("TP") / (F.col("TP") + F.col("FP")))
    )
    .withColumn(
        "recall",
        F.when(F.col("actual_failures") > 0,
               F.col("TP") / F.col("actual_failures"))
    )
    .withColumn(
        "f2",
        F.when(5 * F.col("TP") + 4 * F.col("FN") + F.col("FP") > 0,
               5 * F.col("TP") / (5 * F.col("TP") + 4 * F.col("FN") + F.col("FP")))
    )
    .withColumn("alert_rate", (F.col("TP") + F.col("FP")) / F.col("window_records"))
    .withColumn("window_size_target", F.lit(WINDOW_SIZE))
    .withColumn(
        "status",
        F.when(F.col("window_records") < WINDOW_SIZE, F.lit("WARMING_UP"))
         .when(F.col("actual_failures") < MIN_FAILURES, F.lit("LOW_FAILURE_SUPPORT"))
         .otherwise(F.lit("READY"))
    )
    .withColumn("calculated_at", F.current_timestamp())
)

# Store a historical snapshot only when the latest window/metrics change.
# Re-running unchanged input will NOT insert an identical snapshot.
if not spark.catalog.tableExists(ROLLING_TABLE):
    rolling_df.write.format("delta").mode("error").saveAsTable(ROLLING_TABLE)
else:
    (
        DeltaTable.forName(spark, ROLLING_TABLE).alias("t")
        .merge(
            rolling_df.alias("s"),
            """t.model_uri = s.model_uri
               AND t.threshold = s.threshold
               AND t.latest_udi = s.latest_udi
               AND t.window_end = s.window_end
               AND t.window_records = s.window_records
               AND t.TP = s.TP AND t.FN = s.FN
               AND t.FP = s.FP AND t.TN = s.TN"""
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

print("Rolling metrics snapshots saved.")

# Most recent snapshot per model, rather than all historical snapshots.
latest = Window.partitionBy("model_uri", "threshold").orderBy(
    F.col("calculated_at").desc(), F.col("latest_udi").desc()
)
display(
    spark.table(ROLLING_TABLE)
    .withColumn("rn", F.row_number().over(latest))
    .filter(F.col("rn") == 1)
    .drop("rn")
)
